In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (64283, 66)
           Patient            timestamp  glucose_value meal_type  meal_carbs  \
0  540-ws-training  2027-05-19 11:36:29             76      none         0.0   
1  540-ws-training  2027-05-19 11:41:29             72      none         0.0   
2  540-ws-training  2027-05-19 11:46:29             68      none         0.0   
3  540-ws-training  2027-05-19 11:51:29             65      none         0.0   
4  540-ws-training  2027-05-19 11:56:29             63      none         0.0   

  bolus_type  bolus_dose  basal_value  temp_basal_value exercise_type  ...  \
0     normal         0.8         0.95               0.0          none  ...   
1     normal         0.8         0.95               0.0          none  ...   
2     normal         0.8         0.95               0.0          none  ...   
3     normal         0.8         0.95               0.0          none  ...   
4     normal         0.8         0.95               0.0          none  ...   

   sleep_change gsr_rol

C:\Users\Naveen kumar\AppData\Local\Temp\ipykernel_5664\3682199530.py:4: DtypeWarning: Columns (0: exercise_intensity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
selected_features = [
    # Glucose
    "glucose_value",
    "glucose_prev_5min",
    "glucose_prev_10min",
    "glucose_prev_15min",
    "glucose_prev_30min",
    "glucose_change_5min",
    "glucose_change_10min",
    "glucose_change_15min",
    "glucose_change_30min",
    "glucose_rate_5min",
    "glucose_rate_15min",
    "glucose_rate_30min",
    "glucose_rolling_mean_15min",
    "glucose_rolling_mean_30min",
    "glucose_rolling_std_30min",
    "glucose_min_30min",
    "glucose_max_30min",

    # Insulin
    "bolus_given",
    "bolus_dose_filled",
    "basal_value",
    "basal_rolling_30min",
    "basal_rolling_60min",
    "temp_basal_active",
    "effective_basal",
    "basal_change",

    # Meal
    "meal_event",
    "meal_carbs",

    # Exercise
    "exercise_event",
    "exercise_duration",

    # Time
    "hour",
    "day_of_week",
    "is_weekend"
]

X = df[selected_features].copy()
y = df["glucose_30min_ahead"].copy()

print("Number of features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 32
X shape: (64283, 32)
y shape: (64283,)


In [3]:
missing = X.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

glucose_prev_30min           36
glucose_change_30min         36
glucose_rate_30min           36
glucose_prev_15min           18
glucose_rate_15min           18
glucose_change_15min         18
glucose_change_10min         12
glucose_prev_10min           12
glucose_prev_5min             6
glucose_change_5min           6
glucose_rate_5min             6
glucose_rolling_std_30min     6
dtype: int64


In [4]:
print("Total missing values:", X.isna().sum().sum())

Total missing values: 210


In [5]:
print(X.dtypes)

glucose_value                   int64
glucose_prev_5min             float64
glucose_prev_10min            float64
glucose_prev_15min            float64
glucose_prev_30min            float64
glucose_change_5min           float64
glucose_change_10min          float64
glucose_change_15min          float64
glucose_change_30min          float64
glucose_rate_5min             float64
glucose_rate_15min            float64
glucose_rate_30min            float64
glucose_rolling_mean_15min    float64
glucose_rolling_mean_30min    float64
glucose_rolling_std_30min     float64
glucose_min_30min             float64
glucose_max_30min             float64
bolus_given                     int64
bolus_dose_filled             float64
basal_value                   float64
basal_rolling_30min           float64
basal_rolling_60min           float64
temp_basal_active               int64
effective_basal               float64
basal_change                  float64
meal_event                      int64
meal_carbs  

In [6]:
df = df.sort_values(["Patient", "timestamp"]).reset_index(drop=True)

train_parts = []
test_parts = []

for patient, patient_df in df.groupby("Patient"):
    split_index = int(len(patient_df) * 0.8)

    train_parts.append(patient_df.iloc[:split_index])
    test_parts.append(patient_df.iloc[split_index:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (51424, 66)
Test : (12859, 66)


In [7]:
X_train = train_df[selected_features].copy()
X_test = test_df[selected_features].copy()

y_train = train_df["glucose_30min_ahead"].copy()
y_test = test_df["glucose_30min_ahead"].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (51424, 32)
X_test : (12859, 32)
y_train: (51424,)
y_test : (12859,)


In [8]:
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Missing in X_train:", X_train.isna().sum().sum())
print("Missing in X_test :", X_test.isna().sum().sum())

Missing in X_train: 0
Missing in X_test : 0


In [9]:
X_train_np = X_train.astype("float32").values
X_test_np = X_test.astype("float32").values

y_train_np = y_train.astype("float32").values
y_test_np = y_test.astype("float32").values

print("X_train shape:", X_train_np.shape)
print("Any NaN:", np.isnan(X_train_np).any())
print("Any Inf:", np.isinf(X_train_np).any())

X_train shape: (51424, 32)
Any NaN: False
Any Inf: False


In [10]:
X_train_np = X_train.astype("float32").values
X_test_np = X_test.astype("float32").values

y_train_np = y_train.astype("float32").values
y_test_np = y_test.astype("float32").values

print("X_train shape:", X_train_np.shape)
print("Any NaN:", np.isnan(X_train_np).any())
print("Any Inf:", np.isinf(X_train_np).any())

X_train shape: (51424, 32)
Any NaN: False
Any Inf: False


In [11]:
import tensorflow as tf

normalizer = tf.keras.layers.Normalization()

normalizer.adapt(X_train_np)

print("Normalization complete.")

Normalization complete.


In [12]:
model = tf.keras.Sequential([
    normalizer,

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(16, activation="relu"),

    # Regression output
    tf.keras.layers.Dense(1)
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (51424, 32)            │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65 (264.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 65 (264.00 B)

In [13]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

In [15]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X_train_np,
    y_train_np,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

print("Training complete.")

Epoch 1/50


643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 677.2809 - mae: 19.3783 - val_loss: 713.7099 - val_mae: 20.0527
Epoch 2/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 641.4034 - mae: 18.8098 - val_loss: 851.1736 - val_mae: 22.5623
Epoch 3/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 608.6649 - mae: 18.2969 - val_loss: 860.2263 - val_mae: 22.8246
Epoch 4/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 600.9454 - mae: 18.0883 - val_loss: 1002.2145 - val_mae: 24.6986
Epoch 5/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - loss: 575.5029 - mae: 17.6814 - val_loss: 871.6850 - val_mae: 22.7798
Epoch 6/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - loss: 564.0433 - mae: 17.5268 - val_loss: 846.5080 - val_mae: 22.1025
Epoch 7/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 569.5096 - mae: 17.4483 - val_loss: 816.7844 - val_mae: 21.4957
Epoch 8/50
643/643 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 556.4185 - mae: 17.3234 - val_loss: 918.2061 - val_mae: 23.3037
Epoch 9/50
64

In [ ]:

y_pred = model.predict(X_test_np).flatten()

mae = mean_absolute_error(y_test_np, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_np, y_pred))
r2 = r2_score(y_test_np, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

402/402 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


NameError: name 'mean_absolute_error' is not defined